# Schach-Agent mit Self-Play Reinforcement Learning trainieren

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mark-baumann/schach-reinforcement-lernen/blob/main/notebooks/01_train_chess_rl_agent.ipynb)

Dieses Notebook trainiert ein kleines neuronales Bewertungsnetz (Value Network) für Schachstellungen
per **Self-Play Reinforcement Learning** — das Netz spielt gegen sich selbst, sammelt Partien und
lernt daraus, welche Stellungen gut oder schlecht sind, ganz ohne von Menschen gelabelte Daten.

**Ablauf:**
1. Stellungen werden als 12×8×8-Tensor kodiert (ein Kanal pro Figurentyp/Farbe).
2. Ein kleines CNN (`ValueNet`) sagt für eine Stellung einen Wert in `[-1, 1]` aus Sicht des Spielers am Zug voraus.
3. Im Self-Play wählt das Netz Züge per 1-Ply-Greedy-Suche (mit ε-greedy-Exploration).
4. Als Trainingssignal dient die Monte-Carlo-Rückgabe pro Zug: Materialänderung (Reward Shaping)
   plus ein Bonus/Malus von ±1 am Partieende (Matt gewonnen/verloren).
5. Der Fortschritt wird laufend gegen einen Zufallsspieler evaluiert und geplottet.
6. Das trainierte Modell wird als `.pt`-Datei gespeichert — nutzbar in
   [`02_play_against_agent.ipynb`](./02_play_against_agent.ipynb).

> **Hinweis:** Dies ist ein Lern-/Demo-Projekt, kein State-of-the-Art-Engine wie AlphaZero.
> Mit den Standard-Einstellungen läuft das Training in wenigen Minuten auf der Colab-CPU und
> zeigt einen sichtbaren Lernfortschritt gegen einen Zufallsspieler. Für stärkeres Spiel:
> mehr Self-Play-Partien, tiefere Suche, größeres Netz oder einen Replay-Buffer ergänzen.

**In Google Colab ausführen:** Laufzeit → Laufzeittyp ändern → (optional) GPU wählen, dann alle
Zellen der Reihe nach ausführen (Laufzeit → Alle ausführen).


## 1. Setup

Installiert `python-chess` (in Colab nicht vorinstalliert). PyTorch ist in Colab bereits vorhanden.

In [ ]:
%pip install -q python-chess


In [ ]:
import random
import time

import chess
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Gerät: {device}")

random.seed(0)
torch.manual_seed(0)


## 2. Stellungskodierung und Bewertungsnetz

Jede Stellung wird als 12 Ebenen à 8×8 kodiert: je eine Ebene pro Figurentyp (Bauer, Springer,
Läufer, Turm, Dame, König) und Farbe. Das `ValueNet` ist ein kleines CNN, das daraus einen
Stellungswert in `[-1, 1]` aus Sicht des Spielers am Zug schätzt (`tanh`-Ausgabe).

In [ ]:
PIECE_VALUES = {
    chess.PAWN: 1.0, chess.KNIGHT: 3.0, chess.BISHOP: 3.0,
    chess.ROOK: 5.0, chess.QUEEN: 9.0, chess.KING: 0.0,
}


def material_score(board: chess.Board) -> float:
    """Materialbilanz aus Sicht von Weiß (positiv = Weiß im Vorteil)."""
    score = 0.0
    for _, piece in board.piece_map().items():
        val = PIECE_VALUES[piece.piece_type]
        score += val if piece.color == chess.WHITE else -val
    return score


def encode_board(board: chess.Board) -> torch.Tensor:
    """Kodiert ein Board als 12x8x8-Tensor (One-Hot pro Figurentyp/Farbe)."""
    planes = torch.zeros(12, 8, 8, dtype=torch.float32)
    for sq, piece in board.piece_map().items():
        row, col = divmod(sq, 8)
        idx = (piece.piece_type - 1) + (0 if piece.color == chess.WHITE else 6)
        planes[idx, row, col] = 1.0
    return planes


class ValueNet(nn.Module):
    """Kleines CNN: Stellung -> Wert in [-1, 1] aus Sicht des Spielers am Zug."""

    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(12, 32, 3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, 3, padding=1)
        self.fc1 = nn.Linear(64 * 8 * 8, 128)
        self.fc2 = nn.Linear(128, 1)

    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = x.flatten(1)
        x = F.relu(self.fc1(x))
        return torch.tanh(self.fc2(x)).squeeze(-1)


net = ValueNet().to(device)
print(net)


## 3. Zugauswahl per 1-Ply-Greedy-Suche

Für jeden legalen Zug wird die Folgestellung bewertet; da `ValueNet` immer aus Sicht des
Spielers am Zug bewertet, wird der gegnerische Wert negiert. Gewählt wird der Zug mit dem
höchsten Wert für den ziehenden Spieler.

In [ ]:
@torch.no_grad()
def value_of(board: chess.Board) -> float:
    x = encode_board(board).unsqueeze(0).to(device)
    return net(x).item()


def select_move(board: chess.Board, epsilon: float = 0.0) -> chess.Move:
    legal_moves = list(board.legal_moves)
    if random.random() < epsilon:
        return random.choice(legal_moves)

    best_move, best_value = None, -float("inf")
    for move in legal_moves:
        board.push(move)
        value = -value_of(board)  # Sicht des Gegners -> für uns negieren
        board.pop()
        if value > best_value:
            best_value, best_move = value, move
    return best_move


## 4. Self-Play-Partien erzeugen

Eine Partie wird bis zum Ende (oder `max_plies` Halbzügen) mit ε-greedy-Zugauswahl gespielt.
Als Trainingsziel je Zug dient die Monte-Carlo-Rückgabe: kleine Zwischenbelohnung aus der
Materialänderung dieses Zuges (Reward Shaping, beschleunigt das Lernen deutlich) plus
diskontierte zukünftige Belohnungen, plus ein Bonus von ±1 bei Matt am Partieende.

In [ ]:
def play_self_play_game(epsilon: float = 0.2, max_plies: int = 40, gamma: float = 0.98):
    board = chess.Board()
    states, rewards = [], []
    prev_material = material_score(board)

    for _ in range(max_plies):
        if board.is_game_over():
            break
        mover_is_white = board.turn == chess.WHITE
        states.append(encode_board(board))

        move = select_move(board, epsilon)
        board.push(move)

        new_material = material_score(board)
        delta = new_material - prev_material
        reward = (delta if mover_is_white else -delta) * 0.1  # Reward Shaping
        rewards.append(reward)
        prev_material = new_material

        if board.is_game_over():
            break

    if board.is_checkmate():
        # Der Spieler, der NICHT am Zug ist, hat gerade Matt gesetzt.
        winner_is_white = board.turn == chess.BLACK
        terminal = 1.0 if winner_is_white else -1.0
    else:
        terminal = 0.0  # Remis oder Abbruch nach max_plies

    returns = []
    g = terminal
    for r in reversed(rewards):
        g = r + gamma * g
        returns.append(g)
    returns.reverse()

    # Rückgabe in Sicht des jeweils ziehenden Spielers umrechnen (Weiß beginnt immer).
    perspective_returns = [
        ret if ply_index % 2 == 0 else -ret for ply_index, ret in enumerate(returns)
    ]
    return states, perspective_returns


## 5. Evaluation gegen einen Zufallsspieler

Um den Trainingsfortschritt sichtbar zu machen, lässt das Netz (rein greedy, ohne Exploration)
regelmäßig gegen einen Zufallsspieler antreten. Die Gewinnrate sollte im Trainingsverlauf über
50 % steigen.

In [ ]:
def play_vs_random(net_plays_white: bool, max_plies: int = 80) -> float:
    board = chess.Board()
    for _ in range(max_plies):
        if board.is_game_over():
            break
        is_net_turn = (board.turn == chess.WHITE) == net_plays_white
        if is_net_turn:
            move = select_move(board, epsilon=0.0)
        else:
            move = random.choice(list(board.legal_moves))
        board.push(move)

    if board.is_checkmate():
        white_won = board.turn == chess.BLACK
        net_won = white_won == net_plays_white
        return 1.0 if net_won else 0.0
    return 0.5  # Remis oder kein Ergebnis innerhalb max_plies


def evaluate_win_rate(n_games: int = 20) -> float:
    total = sum(play_vs_random(net_plays_white=(i % 2 == 0)) for i in range(n_games))
    return total / n_games


## 6. Trainingsschleife

Standardmäßig 400 Self-Play-Partien — das dauert auf der Colab-CPU üblicherweise wenige Minuten.
Für ein stärkeres Ergebnis kann `N_GAMES` erhöht werden (z. B. 2000+), am besten mit GPU-Laufzeit.

In [ ]:
N_GAMES = 400
EVAL_EVERY = 50
EPSILON_START, EPSILON_END = 0.3, 0.05

opt = torch.optim.Adam(net.parameters(), lr=1e-3)

win_rates = []  # (game_index, win_rate)
losses = []

start = time.time()
for game_i in range(1, N_GAMES + 1):
    epsilon = EPSILON_START + (EPSILON_END - EPSILON_START) * (game_i / N_GAMES)
    states, returns = play_self_play_game(epsilon=epsilon)
    if not states:
        continue

    X = torch.stack(states).to(device)
    y = torch.tensor(returns, dtype=torch.float32, device=device)

    pred = net(X)
    loss = F.mse_loss(pred, y)

    opt.zero_grad()
    loss.backward()
    opt.step()
    losses.append(loss.item())

    if game_i % EVAL_EVERY == 0:
        wr = evaluate_win_rate(20)
        win_rates.append((game_i, wr))
        elapsed = time.time() - start
        print(f"Partie {game_i:4d}/{N_GAMES} | Loss {loss.item():.4f} | "
              f"Gewinnrate vs. Zufall: {wr:.0%} | {elapsed:.0f}s")

print("Training abgeschlossen.")


## 7. Lernkurve plotten

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(losses)
axes[0].set_title("Trainingsverlust je Partie")
axes[0].set_xlabel("Partie")
axes[0].set_ylabel("MSE-Loss")

if win_rates:
    xs, ys = zip(*win_rates)
    axes[1].plot(xs, ys, marker="o")
axes[1].axhline(0.5, color="gray", linestyle="--", label="Zufallsniveau")
axes[1].set_title("Gewinnrate gegen Zufallsspieler")
axes[1].set_xlabel("Partie")
axes[1].set_ylabel("Gewinnrate")
axes[1].set_ylim(0, 1)
axes[1].legend()

plt.tight_layout()
plt.show()


## 8. Modell speichern

Speichert die Gewichte als `chess_value_net.pt`. In Colab lässt sich die Datei zusätzlich lokal
herunterladen; im Repository liegt sie unter `notebooks/` und kann direkt von
[`02_play_against_agent.ipynb`](./02_play_against_agent.ipynb) geladen werden.

In [ ]:
CHECKPOINT_PATH = "chess_value_net.pt"
torch.save(net.state_dict(), CHECKPOINT_PATH)
print(f"Modell gespeichert unter: {CHECKPOINT_PATH}")

try:
    from google.colab import files
    files.download(CHECKPOINT_PATH)
except ImportError:
    pass  # Läuft nicht in Colab


## 9. Demo: eine Partie ansehen

Lässt das trainierte Netz (Weiß) gegen einen Zufallsspieler (Schwarz) antreten und zeigt die
Endstellung sowie die Zughistorie in algebraischer Notation.

In [ ]:
def play_demo_game(max_plies: int = 80):
    board = chess.Board()
    while not board.is_game_over() and board.fullmove_number <= max_plies:
        if board.turn == chess.WHITE:
            move = select_move(board, epsilon=0.0)
        else:
            move = random.choice(list(board.legal_moves))
        board.push(move)
    return board


demo_board = play_demo_game()

# Zughistorie in algebraischer Notation (SAN) rekonstruieren.
san_moves = []
replay_board = chess.Board()
for move in demo_board.move_stack:
    san_moves.append(replay_board.san(move))
    replay_board.push(move)

print(demo_board)
print()
print("Ergebnis:", demo_board.result())
print("Züge:", " ".join(san_moves))


## Weiterführende Ideen

- **Replay Buffer**: Erfahrungen mehrerer Partien sammeln und daraus mini-batchweise trainieren
  statt nach jeder Partie sofort zu aktualisieren.
- **Tieferes Suchverfahren**: Statt 1-Ply-Greedy z. B. Minimax mit dem `ValueNet` als
  Blattbewertung (ähnlich `app.py`, aber mit gelernter statt handgeschriebener Bewertung) oder
  eine einfache Monte-Carlo-Baumsuche.
- **Policy-Netz ergänzen**: Zusätzlich zur Bewertung auch eine Zugwahrscheinlichkeit lernen
  (AlphaZero-Stil) statt Züge nur über die Bewertungsfunktion zu wählen.
- **Größeres Netz / mehr Partien**: Mit GPU-Laufzeit in Colab lassen sich deutlich mehr
  Self-Play-Partien und ein größeres Netz trainieren.
